#
<h1><span style="color:blue">Evaluating Data Poisoning Attacks</span></h1>

<p><em>Dataset:</em> <a href="https://huggingface.co/datasets/qualifire/prompt-injections-benchmark/viewer/default/test?row=85&views%5B%5D=test" target="_blank">Jailbreak attacks on LLMs</a></p>

#Loading the dataset

Mounts Google Drive to enable file access and persistence.

In [3]:

# Mount Google Drive to read your dataset and save outputs
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

Define the path to the dataset stored on Google Drive

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Evaluating_Data_Poisoning_Attacks/dataset.csv"

Loads the dataset and prints a few samples to inspect labels and text content

In [ ]:
import pandas as pd
df = pd.read_csv(DATASET_PATH)

from textwrap import fill

pd.set_option('display.max_colwidth', None)  # niente "..."
for i, row in df.head(5).iterrows():        # cambia 10 come vuoi
    print(f"#{i}  [{row['label']}]")
    print(fill(str(row['text']), width=100)) # va a capo ogni ~100 caratteri
    print("-" * 80)


Displays the number of rows and columns in the dataset.

In [ ]:
rows, cols = df.shape
print(f"NUmber of rows: {rows} \nNumber of columns: {cols}")

Sets the random seed to ensure reproducibility of experiments.

In [ ]:
random_seed = 50

#Load the multiple choice jailbreak prompts

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount the drive (if you haven't already done so in this session)
drive.mount('/content/drive')

# 2. Define the exact file path
file_path = "/content/drive/MyDrive/Evaluating_Data_Poisoning_Attacks/multiple_choice_cluster.csv"

# 3. Read the CSV file and save it into the variable
multiple_choice_cluster_df = pd.read_csv(file_path)

# 4. Print the first few rows to confirm it was loaded correctly
display(multiple_choice_cluster_df.head())

#training/test/validation split

##entire test set

Splits the dataset into training, validation, and test sets using stratification.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('label', axis=1)
y = df['label']

X_train_temp, X_test_val_temp, y_train, y_test_val = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=random_seed,
    stratify=y
)

X_val_temp, X_test_temp, y_val, y_test = train_test_split(
    X_test_val_temp,
    y_test_val,
    test_size=0.5,
    random_state=random_seed,
    stratify=y_test_val
)

idx_train = X_train_temp.index
idx_val = X_val_temp.index
idx_test = X_test_temp.index

print(f"--- 70% / 15% / 15% ---")
print(f"Training Set (Indexes):   {len(idx_train)} samples")
print(f"Validation Set (Indexes): {len(idx_val)} samples")
print(f"Test Set (Indexes):       {len(idx_test)} samples")

##Multiple choice prompts

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
mc_train_df, mc_test_val_df = train_test_split(
    multiple_choice_cluster_df,
    test_size=0.3,
    random_state=random_seed
)

# 15% validation, 15% test
mc_val_df, mc_test_df = train_test_split(
    mc_test_val_df,
    test_size=0.5,
    random_state=random_seed
)

# Indexes
idx_mc_train = mc_train_df.index
idx_mc_val = mc_val_df.index
idx_mc_test = mc_test_df.index

print("--- Multiple-Choice Split: 70% / 15% / 15% ---")
print(f"Training Set:   {len(idx_mc_train)} samples")
print(f"Validation Set: {len(idx_mc_val)} samples")
print(f"Test Set:       {len(idx_mc_test)} samples")

In [ ]:
print("Train sample:")
display(mc_train_df.head(3))

print("Validation sample:")
display(mc_val_df.head(3))

print("Test sample:")
display(mc_test_df.head(3))

# Attack: Label Flipping

Randomly flips labels in the **training set** (0->1, 1->0).

Four versions:
1. Low % of random labels flipped - entire training set (10%)
2. High % of random labels flipped - entire training set (50%)
3. Low % of labels flipped - multiple-choice prompts only (10%)
4. High % of labels flipped - multiple-choice prompts only (50%)

## Entire training set

### Attack function

In [ ]:
import pandas as pd
import numpy as np
import math

def flip_labels(y_train, flip_rate, random_seed=50):
    """
    Randomly flips a fraction of labels (0->1 and 1->0).
    """
    y_flipped = y_train.copy()
    n = len(y_flipped)
    n_flip = int(math.ceil(n * flip_rate)) if flip_rate > 0 else 0

    rng = np.random.default_rng(random_seed)
    chosen = rng.choice(y_flipped.index.to_numpy(), size=n_flip, replace=False)

    y_flipped.loc[chosen] = 1 - y_flipped.loc[chosen]

    return y_flipped, chosen

### Flip rates

In [ ]:
LOW_RATE  = 0.10  # 10%
HIGH_RATE = 0.50  # 50%

n_train = len(y_train)
print(f'Training set size: {n_train}')
print(f'Low  ({int(LOW_RATE*100)}%): {int(math.ceil(n_train * LOW_RATE))} labels flipped')
print(f'High ({int(HIGH_RATE*100)}%): {int(math.ceil(n_train * HIGH_RATE))} labels flipped')
print(f'\nOriginal label distribution:')
print(y_train.value_counts().to_string())

### Version 1 - Low percentage (10%) of random labels flipped

In [ ]:
y_train_v1, idx_v1 = flip_labels(y_train, LOW_RATE, random_seed)

print(f'Version 1 - labels flipped: {len(idx_v1)}')
print(f'New label distribution:')
print(y_train_v1.value_counts().to_string())

### Version 2 - High percentage (50%) of random labels flipped

In [ ]:
y_train_v2, idx_v2 = flip_labels(y_train, HIGH_RATE, random_seed)

print(f'Version 2 - labels flipped: {len(idx_v2)}')
print(f'New label distribution:')
print(y_train_v2.value_counts().to_string())

### Inspect some flipped samples

In [ ]:
from textwrap import fill

def print_flipped_samples(X_train, y_original, y_flipped, flipped_idx, num_samples=5):
    sample = flipped_idx[:num_samples]
    for i, idx in enumerate(sample):
        orig = y_original.loc[idx]
        new  = y_flipped.loc[idx]
        print(f'--- Sample {i+1} | label: {orig} -> {new} ---')
        print(fill(str(X_train.loc[idx, 'text']), width=100))
        print()

print('=== Version 1 (low %) - flipped samples ===')
print_flipped_samples(X_train_temp, y_train, y_train_v1, idx_v1)

---
## Multiple-choice jailbreak prompts
Same attack but only within the MC training set. Uses `true_label` column.

In [ ]:
X_mc_train = mc_train_df.drop('true_label', axis=1)
y_mc_train = mc_train_df['true_label']

n_mc = len(y_mc_train)
print(f'MC training set size: {n_mc}')
print(f'Low  ({int(LOW_RATE*100)}%): {int(math.ceil(n_mc * LOW_RATE))} labels flipped')
print(f'High ({int(HIGH_RATE*100)}%): {int(math.ceil(n_mc * HIGH_RATE))} labels flipped')
print(f'\nOriginal MC label distribution:')
print(y_mc_train.value_counts().to_string())

### Version 3 - Low percentage (10%) of MC labels flipped

In [ ]:
y_mc_train_v3, idx_mc_v3 = flip_labels(y_mc_train, LOW_RATE, random_seed)

print(f'Version 3 - MC labels flipped: {len(idx_mc_v3)}')
print(f'New MC label distribution:')
print(y_mc_train_v3.value_counts().to_string())

### Version 4 - High percentage (50%) of MC labels flipped

In [ ]:
y_mc_train_v4, idx_mc_v4 = flip_labels(y_mc_train, HIGH_RATE, random_seed)

print(f'Version 4 - MC labels flipped: {len(idx_mc_v4)}')
print(f'New MC label distribution:')
print(y_mc_train_v4.value_counts().to_string())

### Inspect some flipped MC samples

In [ ]:
print('=== Version 3 (low % MC) - flipped samples ===')
print_flipped_samples(X_mc_train, y_mc_train, y_mc_train_v3, idx_mc_v3)

---
## Summary

In [ ]:
summary = [
    ('Version 1', 'Entire training set', f'{int(LOW_RATE*100)}%',  len(idx_v1),    len(y_train)),
    ('Version 2', 'Entire training set', f'{int(HIGH_RATE*100)}%', len(idx_v2),    len(y_train)),
    ('Version 3', 'Multiple-choice',     f'{int(LOW_RATE*100)}%',  len(idx_mc_v3), len(y_mc_train)),
    ('Version 4', 'Multiple-choice',     f'{int(HIGH_RATE*100)}%', len(idx_mc_v4), len(y_mc_train)),
]

print(f'{"Version":<12}{"Subset":<22}{"Rate":<8}{"Flipped":<10}{"Set size"}')
print('-' * 60)
for v, subset, rate, flipped, total in summary:
    print(f'{v:<12}{subset:<22}{rate:<8}{flipped:<10}{total}')